In [2]:
import pandas as pd
import numpy as np
import cobra
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
import working_w_seed_models as wm
from configparser import ConfigParser
import importlib
importlib.reload(wm)


<module 'working_w_seed_models' from '/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py'>

In [3]:
config = ConfigParser()
config.read("build_pbi_model.ini")
tmp = cobra.io.load_json_model("../../results/pbi_model_gapfill/bifermentans_gapfilled_from_cdiff_metabolomics.json")
model = wm.Model(tmp, config)
tmp = cobra.io.load_json_model("../../data/cdiff/icdf843_2_unidirectional.json")
cdiff_model = wm.Model(tmp, config)

/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:42: DtypeWarning: Columns (1,3,4,6,7,8,9,11,12,13,16,17,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  self.db = pd.read_csv(database_path)
/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:1368: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  db_compound = pd.read_csv(compounds_filepath, sep = "\t", index_col = 0)


Length of conversions after 1st method:  721


/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:1389: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  compounds = pd.read_csv(compounds_filepath, sep = "\t").set_index("name")


Length of conversions after 2nd method:  806
Length of conversions after 3rd method:  891


/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:42: DtypeWarning: Columns (1,3,4,6,7,8,9,11,12,13,16,17,19,20,21,22,23,24) have mixed types. Specify dtype option on import or set low_memory=False.
  self.db = pd.read_csv(database_path)
/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:1368: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  db_compound = pd.read_csv(compounds_filepath, sep = "\t", index_col = 0)


Length of conversions after 1st method:  721


/home/christine/Partners HealthCare Dropbox/Christine Tataru/metabolic_modeling/scripts/metabolic_models/working_w_seed_models.py:1389: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  compounds = pd.read_csv(compounds_filepath, sep = "\t").set_index("name")


Length of conversions after 2nd method:  806
Length of conversions after 3rd method:  891


In [17]:
def check_biomass_reactions(model, cdiff_model):
    # we just use the cdiff_model to get conversions, b/c it's hard to interpret the modelSEED ids
    model.enableTransportReactions()
    model.model.objective = "Bio_DNA"
    model.addSink(cdiff_model.conversions["DNA_c"])
    if model.model.optimize().objective_value > 0:
        print("\n Model is able to produce DNA!")
    else:
        print("Model IS NOT able to produce DNA")


    model.enableTransportReactions()
    model.model.objective = "Bio_RNA"
    model.addSink(cdiff_model.conversions["RNA_c"])
    if model.model.optimize().objective_value > 0:
        print("\n Model is able to produce RNA!")
    else:
        print("Model IS NOT able to produce RNA")


    model.model.objective = "Bio_CW"
    model.addSink(cdiff_model.conversions["CW_c"])
    if model.model.optimize().objective_value > 0:
        print("\n Model is able to produce CW!")
    else:
        print("Model IS NOT able to produce CW")

    model.model.objective = "Bio_prot"
    model.addSink(cdiff_model.conversions["Prot_c"])
    if model.model.optimize().objective_value > 0:
        print("\n Model is able to produce Protein!")
    else:
        print("Model IS NOT able to produce Protein")

    model.model.objective = "Bio_lip"
    model.addSink(cdiff_model.conversions["Lip_c"])
    if model.model.optimize().objective_value > 0:
        print("\n Model is able to produce lipid!")
    else:
        print("Model IS NOT able to produce lipid")

    model.model.objective = "Bio_SPs"
    model.addSink(cdiff_model.conversions["SPs_c"])
    if model.model.optimize().objective_value > 0:
        print("\n Model is able to produce solute pool!")
    else:
        print("Model IS NOT able to produce solute pool")

model.model.objective = "bio1"
check_biomass_reactions(model, cdiff_model)


 Model is able to produce DNA!

 Model is able to produce RNA!

 Model is able to produce CW!

 Model is able to produce Protein!

 Model is able to produce lipid!

 Model is able to produce solute pool!


In [75]:
# Check reductive leucine metabolism
model.enableTransportReactions()
model.checkReactionLinked('ID_382', ['cpd00107_e0', 'cpd00084_e0', model.conversions['pntoR_c']], concentrations=np.arange(0, 1, 0.1)) # leucine and L-Cysteine and pantothenate
# metabolite list is changing concentrations, reaction is changing/not changing flux in response to change in concentration of metabolite

# after this point, I need to find another dependence, which is not worth doing manually


Testing reaction link to substrate
Biomass:  999.9999999999999
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Biomass:  1000.0
Reaction flux independent of substrate concentration


In [73]:
# check proline racemase
model.checkReactionLinked('rxn00933_c0', [model.conversions['proL_e']], concentrations=np.arange(0, 1, 0.1))

Testing reaction link to substrate
Biomass:  9.999999999999998
Biomass:  10.1
Biomass:  10.2
Biomass:  10.299999999999997
Biomass:  10.399999999999999
Biomass:  10.499999999999998
Biomass:  10.6
Biomass:  10.699999999999998
Biomass:  10.799999999999999
Biomass:  10.899999999999999
Reaction flux is correlated is substrate concentration


In [77]:
cobra.io.write_sbml_model(model.model,  "../../results/pbi_model_gapfill/bifermentans_gapfilled_from_cdiff_metabolomics.xml")

123 metabolites are utilized in the model, but not produced.

In [122]:
from cobra import Reaction
metabolites_not_produced = ["cpd04097_c0","cpd15747_c0","cpd15308_c0","cpd01012_c0","cpd15352_c0","cpd03915_c0","cpd03163_c0","cpd00822_c0","cpd00507_c0","cpd00790_c0","cpd15752_c0","cpd00666_c0","cpd01571_c0","cpd01405_c0","cpd04122_c0","cpd15751_c0","cpd03447_c0","cpd01572_c0","cpd03052_c0","cpd15706_c0","cpd15345_c0","cpd15746_c0","cpd15754_c0","cpd11825_c0","cpd00738_c0","cpd03706_c0","cpd00305_c0","cpd00139_c0","cpd15753_c0","cpd03422_c0","cpd02524_c0","cpd15338_c0","cpd00158_c0","cpd00044_c0","cpd00263_c0","cpd03490_c0","cpd00419_c0","cpd01157_c0","cpd01078_c0","cpd01570_c0","cpd09410_c0","cpd00337_c0","cpd03702_c0","cpd00938_c0","cpd15661_c0","cpd00434_c0","cpd00291_c0","cpd01727_c0","cpd15749_c0","cpd15362_c0","cpd15306_c0","cpd00120_c0","cpd15277_c0","cpd00490_c0","cpd00620_c0","cpd02507_c0","cpd02039_c0","cpd02279_c0","cpd01107_c0","cpd00673_c0","cpd00408_c0","cpd00424_c0","cpd00742_c0","cpd15750_c0","cpd15748_c0","cpd15310_c0","cpd21480_c0","cpd14958_c0","cpd02737_c0","cpd15807_c0","cpd21087_c0","cpd19020_c0","cpd21041_c0","cpd00531_c0","cpd00109_c0","cpd01101_c0","cpd00858_c0","cpd09027_c0","cpd01550_c0","cpd02074_c0","cpd02414_c0","cpd08625_c0","cpd02605_c0","cpd02692_c0","cpd27569_c0","M-cbp-c_c0","cpd00551_c0","M-ctnycoa-c_c0","M-fru1p-c_c0","cpd19009_c0","cpd00709_c0","M-glyc-c_c0","cpd01775_c0","cpd00807_c0","M-fes-c_c0","M-aacp-c_c0","M-rl5p-c_c0","cpd00911_c0","M-2drib1p-c_c0","cpd03385_c0","cpd26854_c0","cpd01882_c0","cpd03828_c0","M-2m3hbcoa-c_c0","cpd00760_c0","cpd00658_c0","cpd15565_c0","cpd12783_c0","cpd03396_c0","cpd01394_c0","cpd12369_c0","cpd02763_c0","cpd22234_c0","cpd27378_c0","cpd03561_c0","cpd02461_c0","cpd02608_c0","cpd00876_c0","cpd01587_c0","cpd00351_c0","cpd02522_c0","cpd03065_c0","cpd11609_c0"]
ids = []
names = []
for metabolite in metabolites_not_produced:
    met = model.model.metabolites.get_by_id(metabolite)
    ids.append(met.id)
    names.append(met.name)

not_produced = pd.DataFrame({"id": ids, "name": names})

# cpd01012_c0: Cadmium
reaction = Reaction("rxn_import_cpd01012_c0", 
                                      name="Import Cadmium", 
                                      lower_bound=0, 
                                      upper_bound=1000)
reaction.add_metabolites({model.model.metabolites.get_by_id("cpd01012_c0"): 1, model.model.metabolites.get_by_id("cpd01012_e0"): -1})
model.model.add_reactions([reaction])

not_produced.iloc[0:20, :]

Ignoring reaction 'rxn_import_cpd01012_c0' since it already exists.


,id,name
0,cpd04097_c0,Pb [c0]
1,cpd15747_c0,"Myristoyllipoteichoic acid (n=24), linked, uns..."
2,cpd15308_c0,"1,2-Diacyl-sn-glycerol ditetradec-7-enoyl [c0]"
3,cpd01012_c0,Cd2+ [c0]
4,cpd15352_c0,2-Demethylmenaquinone 8 [c0]
5,cpd03915_c0,Cob(I)yrinate diamide [c0]
6,cpd03163_c0,Selenomethionine [c0]
7,cpd00822_c0,O-Succinyl-L-homoserine [c0]
8,cpd00507_c0,Glycerophosphocholine [c0]
9,cpd00790_c0,O-Acetyl-L-homoserine [c0]


125 metabolites are produced by not utiliezd


In [123]:
metabolites_not_utilized = ["cpd03524_c0","cpd00477_c0","cpd04097_e0","cpd03289_c0","cpd15774_c0","cpd00793_c0","cpd01012_e0","cpd00112_c0","cpd15239_c0","cpd15499_c0","cpd15353_c0","cpd03392_c0","cpd00244_c0","cpd11578_c0","cpd00135_c0","cpd15779_c0","cpd00300_c0","cpd02590_c0","cpd03454_c0","cpd14955_c0","cpd11579_c0","cpd00210_c0","cpd00532_c0","cpd00318_c0","cpd01441_c0","cpd15778_c0","cpd03448_c0","cpd03724_c0","cpd00989_c0","cpd01831_c0","cpd15773_c0","cpd00359_c0","cpd15781_c0","cpd03161_c0","cpd00073_c0","cpd01553_c0","cpd00374_c0","cpd03048_c0","cpd11596_c0","cpd15780_c0","cpd03914_c0","cpd00988_c0","cpd01401_c0","cpd01720_c0","cpd03701_c0","cpd02255_c0","cpd15663_c0","cpd11575_c0","cpd02357_c0","cpd09878_c0","cpd15776_c0","cpd03424_c0","cpd00579_c0","cpd11462_c0","cpd03326_c0","cpd08023_c0","cpd02886_c0","cpd00425_c0","cpd03034_c0","cpd15777_c0","cpd06227_c0","cpd15775_c0","cpd21497_c0","cpd19019_c0","cpd00895_c0","cpd27020_c0","cpd01042_c0","cpd03091_c0","cpd14865_c0","cpd00531_e0","cpd00110_c0","cpd01567_c0","cpd00585_c0","cpd00922_c0","cpd10516_c0","cpd02333_c0","cpd01298_c0","cpd11416_c0","cpd00021_c0","cpd00361_c0","M-2but-c_c0","cpd02053_c0","cpd01777_c0","cpd00641_c0","M-isocap-c_c0","cpd00840_c0","M-fesm-c_c0","cpd00284_c0","M-aimzcrnt-c_c0","cpd03705_c0","M-ncmylasp-c_c0","M-3uprop-c_c0","M-casp-c_c0","cpd00122","cpd00441_c0","M-isop-c_c0","M-isobutp-c_c0","cpd02124_c0","cpd02125_c0","cpd00791_c0","cpd00168_c0","cpd19177_c0","cpd00222_c0","cpd16579_c0","cpd11451_c0","cpd03387_c0","cpd00211_e0","cpd12223_c0","cpd15596_c0","cpd03300_c0","cpd00703_c0","cpd01546_c0","cpd00912_c0","cpd02416_c0","cpd00767_c0","cpd24480_c0","cpd01921_c0","cpd27614_c0","cpd02315_c0","cpd00489_c0","cpd00308_c0","cpd01468_c0","cpd00323_c0","cpd02235_c0","cpd11610_c0"]
ids = []
names = []
for metabolite in metabolites_not_utilized:
    met = model.model.metabolites.get_by_id(metabolite)
    ids.append(met.id)
    names.append(met.name)

not_utilized = pd.DataFrame({"id": ids, "name": names})
not_utilized.iloc[0:20, :]

,id,name
0,cpd03524_c0,10-Formyl-THF-L-glutamate [c0]
1,cpd00477_c0,N-Acetyl-L-glutamate [c0]
2,cpd04097_e0,Pb [e0]
3,cpd03289_c0,L-2-Acetamido-6-oxopimelate [c0]
4,cpd15774_c0,"Myristoyllipoteichoic acid (n=24), linked, D-a..."
5,cpd00793_c0,Thiamine phosphate [c0]
6,cpd01012_e0,Cd2+ [e0]
7,cpd00112_c0,CMP-N-acetylneuraminate [c0]
8,cpd15239_c0,Hexadecenoyl-ACP [c0]
9,cpd15499_c0,Menaquinol 8 [c0]


,id,name
0,cpd04097_c0,Pb [c0]
1,cpd15747_c0,"Myristoyllipoteichoic acid (n=24), linked, uns..."
2,cpd15308_c0,"1,2-Diacyl-sn-glycerol ditetradec-7-enoyl [c0]"
3,cpd01012_c0,Cd2+ [c0]
4,cpd15352_c0,2-Demethylmenaquinone 8 [c0]
5,cpd03915_c0,Cob(I)yrinate diamide [c0]
6,cpd03163_c0,Selenomethionine [c0]
7,cpd00822_c0,O-Succinyl-L-homoserine [c0]
8,cpd00507_c0,Glycerophosphocholine [c0]
9,cpd00790_c0,O-Acetyl-L-homoserine [c0]
